# Exploring one race: 2025 Italian Grand Prix (Monza)

Goal: understand the structure and quality of the OpenF1 data before automating the download of the whole seasons.

Data source: [OpenF1](https://openf1.org), free historical data from 2023 onwards.

In [7]:
import pandas as pd
import requests

BASE_URL = "https://api.openf1.org/v1"

response = requests.get(
    f"{BASE_URL}/sessions",
    params={"year": 2025, "location": "Monza", "session_name": "Race"},
    timeout=30,
)
response.raise_for_status()

sessions = pd.DataFrame(response.json())
sessions

,session_key,session_type,session_name,date_start,date_end,meeting_key,circuit_key,circuit_short_name,country_key,country_code,country_name,location,gmt_offset,year,is_cancelled
0,9912,Race,Race,2025-09-07T13:00:00+00:00,2025-09-07T15:00:00+00:00,1268,39,Monza,13,ITA,Italy,Monza,02:00:00,2025,False


In [8]:
SESSION_KEY = 9912

def get(endpoint, **params):
    response = requests.get(f"{BASE_URL}/{endpoint}", params=params, timeout=30)
    response.raise_for_status()
    return pd.DataFrame(response.json())


laps = get("laps", session_key=SESSION_KEY)
print(laps.shape)
laps.head()

(976, 16)


,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed
0,1268,9912,1,1,2025-09-07T13:03:34.805000+00:00,29.893,29.088,28.517,321.0,309.0,False,87.498,"[2048, 2049, 2049, 2049, 2049, 2049]","[2051, 2049, 2049, 2051, 2051, 2049, 2049]","[2051, 2049, 2049, 2049, 2049, 2049, 2049, 204...",284.0
1,1268,9912,4,1,2025-09-07T13:03:34.805000+00:00,30.109,29.911,27.903,330.0,312.0,False,87.923,"[2048, 2049, 2051, 2049, 2049, 2049]","[2049, 2049, 2049, 2049, 2049, 2049, 2049]","[2049, 2051, 2049, 2049, 2051, 2051, 2049, 205...",267.0
2,1268,9912,16,1,2025-09-07T13:03:34.805000+00:00,30.132,30.558,28.013,339.0,318.0,False,88.703,"[2048, 2049, 2049, 2049, 2049, 2049]","[2049, 2049, 2049, 2049, 2049, 2051, 2049]","[2049, 2049, 2049, 2051, 2049, 2049, 2051, 205...",292.0
3,1268,9912,81,1,2025-09-07T13:03:34.805000+00:00,30.184,30.023,28.116,322.0,318.0,False,88.323,"[2048, 2049, 2049, 2049, 2049, 2049]","[2049, 2049, 2049, 2049, 2049, 2049, 2049]","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...",287.0
4,1268,9912,63,1,2025-09-07T13:03:34.805000+00:00,30.785,30.234,28.243,328.0,319.0,False,89.262,"[2048, 2049, 2049, 2049, 2049, 2049]","[2049, 2049, 2049, 2049, 2049, 2049, 2049]","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...",286.0


In [9]:
stints = get("stints", session_key=SESSION_KEY)
print(stints.shape)
stints.sort_values(["driver_number", "stint_number"]).head(10)

(39, 8)


,meeting_key,session_key,stint_number,driver_number,lap_start,lap_end,compound,tyre_age_at_start
11,1268,9912,1,1,1,37,MEDIUM,0
31,1268,9912,2,1,38,53,HARD,0
3,1268,9912,1,4,1,46,MEDIUM,0
35,1268,9912,2,4,47,53,SOFT,2
8,1268,9912,1,5,1,20,MEDIUM,0
24,1268,9912,2,5,21,53,HARD,0
10,1268,9912,1,6,1,32,HARD,1
28,1268,9912,2,6,33,53,MEDIUM,0
5,1268,9912,1,10,1,49,HARD,0
36,1268,9912,2,10,50,52,SOFT,0


In [10]:
stint_cols = ["driver_number", "stint_number", "lap_start", "lap_end", "compound", "tyre_age_at_start"]

laps_tyres = laps.merge(stints[stint_cols], on="driver_number")

in_stint = (laps_tyres["lap_number"] >= laps_tyres["lap_start"]) & (
    laps_tyres["lap_number"] <= laps_tyres["lap_end"]
)
laps_tyres = laps_tyres[in_stint].copy()

laps_tyres["tyre_age"] = (
    laps_tyres["tyre_age_at_start"] + laps_tyres["lap_number"] - laps_tyres["lap_start"]
)

print("Giri originali:", len(laps), "| Giri con gomma:", len(laps_tyres))

cols = ["lap_number", "lap_duration", "compound", "stint_number", "tyre_age", "is_pit_out_lap"]
laps_tyres.loc[
    (laps_tyres["driver_number"] == 1) & laps_tyres["lap_number"].between(34, 42), cols
]

Giri originali: 976 | Giri con gomma: 975


,lap_number,lap_duration,compound,stint_number,tyre_age,is_pit_out_lap
1239,34,82.795,MEDIUM,1,33,False
1275,35,82.651,MEDIUM,1,34,False
1309,36,82.699,MEDIUM,1,35,False
1345,37,87.432,MEDIUM,1,36,False
1384,38,102.349,HARD,2,0,True
1424,39,81.402,HARD,2,1,False
1460,40,81.625,HARD,2,2,False
1496,41,81.358,HARD,2,3,False
1532,42,81.445,HARD,2,4,False


In [11]:
check = laps.merge(
    laps_tyres[["driver_number", "lap_number"]],
    on=["driver_number", "lap_number"],
    how="left",
    indicator=True,
)
orphans = check[check["_merge"] == "left_only"]
display(orphans[["driver_number", "lap_number", "lap_duration", "is_pit_out_lap"]])

stints[stints["driver_number"].isin(orphans["driver_number"])].sort_values("stint_number")

,driver_number,lap_number,lap_duration,is_pit_out_lap
475,14,25,NaN,False


,meeting_key,session_key,stint_number,driver_number,lap_start,lap_end,compound,tyre_age_at_start
2,1268,9912,1,14,1,20,MEDIUM,1
23,1268,9912,2,14,21,24,HARD,1


In [12]:
pit = get("pit", session_key=SESSION_KEY)
print(pit.shape)
pit.sort_values("lap_number")

(19, 8)


,date,session_key,lap_number,meeting_key,driver_number,lane_duration,pit_duration,stop_duration
0,2025-09-07T13:16:53.530000+00:00,9912,9,1268,30,24.156,24.156,2.2
1,2025-09-07T13:29:32.907000+00:00,9912,18,1268,87,25.372,25.372,3.3
2,2025-09-07T13:30:55.654000+00:00,9912,19,1268,22,24.372,24.372,2.3
3,2025-09-07T13:32:17.708000+00:00,9912,20,1268,14,25.160,25.160,3.2
4,2025-09-07T13:32:18.150000+00:00,9912,20,1268,5,26.143,26.143,4.0
5,2025-09-07T13:41:53.352000+00:00,9912,27,1268,63,25.021,25.021,2.9
6,2025-09-07T13:43:27.992000+00:00,9912,28,1268,12,24.916,24.916,2.9
7,2025-09-07T13:46:20.012000+00:00,9912,30,1268,55,24.605,24.605,3.0
8,2025-09-07T13:49:09.846000+00:00,9912,32,1268,6,24.326,24.326,2.2
10,2025-09-07T13:50:45.890000+00:00,9912,33,1268,43,24.971,24.971,2.8


In [13]:
race_control = get("race_control", session_key=SESSION_KEY)
print(race_control.shape)

penalties = race_control[race_control["message"].str.contains("PENALTY", case=False, na=False)]
penalties[["lap_number", "driver_number", "category", "message"]]

(73, 11)


,lap_number,driver_number,category,message
31,9,NaN,Other,FIA STEWARDS: 5 SECOND TIME PENALTY FOR CAR 31...
61,46,NaN,Other,FIA STEWARDS: 10 SECOND TIME PENALTY FOR CAR 8...
66,53,NaN,Other,FIA STEWARDS: 5 SECOND TIME PENALTY FOR CAR 12...
71,53,NaN,Other,FIA STEWARDS: PENALTY SERVED - 5 SECOND TIME P...


In [14]:
pd.set_option("display.max_colwidth", None)
penalties[["lap_number", "category", "message"]]

,lap_number,category,message
31,9,Other,FIA STEWARDS: 5 SECOND TIME PENALTY FOR CAR 31 (OCO) - FORCING ANOTHER DRIVER OFF THE TRACK
61,46,Other,FIA STEWARDS: 10 SECOND TIME PENALTY FOR CAR 87 (BEA) - CAUSING A COLLISION
66,53,Other,FIA STEWARDS: 5 SECOND TIME PENALTY FOR CAR 12 (ANT) - DRIVING ERRATICALLY
71,53,Other,FIA STEWARDS: PENALTY SERVED - 5 SECOND TIME PENALTY FOR CAR 31 (OCO) - FORCING ANOTHER DRIVER OFF THE TRACK


In [15]:
drivers = get("drivers", session_key=SESSION_KEY)
drivers[["driver_number", "name_acronym", "full_name", "team_name", "team_colour"]].sort_values(
    "driver_number"
)

,driver_number,name_acronym,full_name,team_name,team_colour
0,1,VER,Max VERSTAPPEN,Red Bull Racing,4781D7
1,4,NOR,Lando NORRIS,McLaren,F47600
2,5,BOR,Gabriel BORTOLETO,Kick Sauber,01C00E
3,6,HAD,Isack HADJAR,Racing Bulls,6C98FF
4,10,GAS,Pierre GASLY,Alpine,00A1E8
5,12,ANT,Kimi ANTONELLI,Mercedes,00D7B6
6,14,ALO,Fernando ALONSO,Aston Martin,229971
7,16,LEC,Charles LECLERC,Ferrari,ED1131
8,18,STR,Lance STROLL,Aston Martin,229971
9,22,TSU,Yuki TSUNODA,Red Bull Racing,4781D7


In [ ]:
## Findings

- **Laps** (976 rows): one row per driver and lap. No tyre information.
- **Stints** (39 rows): compound and tyre age per stint. Joining laps and stints gives compound and tyre age for every lap. 975 of 976 laps matched: the missing one is Alonso's unfinished lap 25 (retirement, no lap time).
- **Pit** (19 rows): consistent with stints (39 stints − 20 drivers = 19 stops). `lap_number` is the in-lap. `lane_duration` is the time lost in the pit lane, `stop_duration` the stationary time.
- **Race control**: penalties have no `driver_number`; the car number must be parsed from the message. Ocon's 8.6 s stop includes a 5 s penalty. Norris (5.9 s) and Stroll (16.2 s) had no penalty: genuine slow stops, to keep.
- **Drivers**: teammates share the team colour. Driver numbers and team names can change between seasons.

## Cleaning rules

- Drop laps without `lap_duration`.
- Flag in-laps (from the pit table) and out-laps (`is_pit_out_lap`) and exclude them from tyre degradation analysis.
- Treat lap 1 separately (standing start).
- Remove served penalties from `stop_duration` before comparing pit crews.